In [1]:
# CELL 1: Setup Awal dan Install Library
!pip install pyspark Sastrawi nltk pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.6 MB/s eta 0:00:00


In [2]:
# CELL 2: Inisialisasi Spark Session (Distributed Core)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, udf
from pyspark.sql.types import StringType, DoubleType
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, Normalizer

spark = SparkSession.builder \
    .appName("DTS_Distributed_Recommender") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

print("Spark Session berhasil dijalankan secara terdistribusi!")

Spark Session berhasil dijalankan secara terdistribusi!


In [9]:
# CELL 3: Pengunduhan Dataset Otomatis (Film & Buku)
import os, requests
def download_data():
    urls = {
        "movies": "https://raw.githubusercontent.com/rengalv/Movies-Data-Analysis-Grab-a-Popcorn/master/tmdb_5000_movies.csv",
        "books": "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv"
    }
    for name, url in urls.items():
        if not os.path.exists(f"{name}.csv"):
            print(f"Downloading {name} dataset...")
            r = requests.get(url)
            with open(f"{name}.csv", 'wb') as f: f.write(r.content)
    print("Dataset Film dan Buku siap digunakan!")

download_data()

# PERBAIKAN: Menambahkan parameter quote dan escape
df_movies_spark = spark.read.csv("movies.csv", header=True, inferSchema=True, quote='"', escape='"')
df_books_spark = spark.read.csv("books.csv", header=True, inferSchema=True, quote='"', escape='"')

# PREVIEW DATA (Menampilkan 5 data teratas)
print("=== 5 DATA FILM ===")
df_movies_spark.select("title", "genres", "vote_average").show(5, truncate=False)

print("\n=== 5 DATA BUKU ===")
df_books_spark.select("title", "authors", "average_rating").show(5, truncate=False)


Dataset Film dan Buku siap digunakan!
=== 5 DATA FILM ===
+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------+------------+
|title                                   |genres                                                                                                                                |vote_average|
+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------+------------+
|Avatar                                  |[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]|7.2         |
|Pirates of the Caribbean: At World's End|[{"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 28, "name": "Action"}]                                        |6

In [12]:
# CELL 4: Distributed Preprocessing (Spark UDF)
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from pyspark.sql.functions import trim # Tambahan Import

stop_words = list(set(stopwords.words('english')))
ps = PorterStemmer()

def clean_text_func(text):
    if text is None: return ""
    text = text.lower()
    text = "".join([c for c in text if c.isalnum() or c.isspace()])
    words = [ps.stem(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

clean_text_udf = udf(clean_text_func, StringType())

print("Memulai Preprocessing Terdistribusi...")
# Gunakan trim() agar tidak ada spasi tersembunyi pada judul
df_movies_proc = df_movies_spark.select(trim(col("title")).alias("title"), "overview") \
    .dropna(subset=["overview"]) \
    .withColumn("cleaned_text", clean_text_udf(col("overview")))

# Terapkan juga untuk dataset Buku
df_books_proc = df_books_spark.select(trim(col("title")).alias("title"), "authors") \
    .dropna() \
    .withColumn("cleaned_text", clean_text_udf(col("title")))

print("Preprocessing selesai!")


Memulai Preprocessing Terdistribusi...
Preprocessing selesai!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
# CELL 5: Implementasi TF-IDF Terdistribusi Asli
def run_distributed_tfidf(df):
    tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words")
    words_df = tokenizer.transform(df)
    hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=5000)
    featurized_df = hashingTF.transform(words_df)
    idf = IDF(inputCol="rawFeatures", outputCol="features")
    idf_model = idf.fit(featurized_df)
    rescaled_df = idf_model.transform(featurized_df)
    normalizer = Normalizer(inputCol="features", outputCol="normFeatures", p=2.0)
    return normalizer.transform(rescaled_df)

movies_final = run_distributed_tfidf(df_movies_proc).cache()

In [15]:
# CELL 6: Fungsi Rekomendasi (Similarity Engine Parallel)
def get_recommendations(df_final, target_title, top_n=5):
    target_row = df_final.filter(lower(col("title")) == target_title.lower()).select("normFeatures").collect()
    if not target_row: return "Not Found"
    target_vec = target_row[0][0]
    dot_product_udf = udf(lambda v: float(v.dot(target_vec)), DoubleType())
    sim_df = df_final.withColumn("similarity", dot_product_udf(col("normFeatures")))
    return sim_df.filter(lower(col("title")) != target_title.lower()) \
                    .select("title", "similarity") \
                    .orderBy(col("similarity").desc()) \
                    .limit(top_n).toPandas()

get_recommendations(movies_final, "The Great Gatsby")

,title,similarity
0,The True Story of Puss 'n Boots,0.163541
1,Memoirs of an Invisible Man,0.158857
2,Love Letters,0.143919
3,The Road,0.134762
4,Rabbit Hole,0.133941
